In [37]:
import pandas as pd
from dotenv import dotenv_values
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
import os
from groq import Groq
from dotenv import dotenv_values
from pprint import pprint
from google import genai
from google.genai import types

In [2]:
# Load Env Varaibles + Import Function Help Us to Embeding Documents
config = dotenv_values(".env")
GROQ_API_KEY = config["GROQ_API_KEY"]
GEMENI_API_KEY = config["GEMENI_API_KEY"]

In [18]:
# try Gemeni Embedding Function
gemeni_client = genai.Client(api_key=GEMENI_API_KEY)
result = gemeni_client.models.embed_content(
        model="gemini-embedding-2",
        contents="What is the meaning of life?"
)
print(result.embeddings[0].values)

[-0.016332133, -0.0043764366, -0.0011324773, -0.011240026, 0.00029597108, 0.0014934157, -0.018807797, 0.013529229, 0.019723475, -0.049145423, -0.020023376, 0.019897161, -0.025475917, -0.016769981, 0.011656811, 0.014576932, 0.017489318, 0.0019741745, -0.033542495, -0.0052898847, -0.01738165, -0.0071869395, 0.005816727, 0.0036677625, -0.004281101, 0.010329708, -0.005853738, 0.007902185, -0.0031149625, 0.13209803, -0.016384594, 0.010829647, -0.00854677, 0.0034034406, 0.0017756857, 0.015020709, 0.00256653, -0.00677215, 0.014306844, -0.010287376, 0.009519303, 0.010457309, 0.009206434, 0.012812066, -0.010743783, -0.00047993741, -0.020863462, -0.005375269, 0.0116195325, 0.00081031286, -0.006415543, 0.024154454, -0.008800871, -0.039657805, -0.0070156506, -0.013774676, 0.0153811, -0.0025023571, 0.013820424, 0.018328026, -0.025549209, 0.0067770947, 0.0013363153, 0.029920602, -0.019204818, -0.007714158, 0.020704817, -0.029377379, 0.026432242, -0.0097309025, 0.004420867, 0.0077373628, -0.025789557

In [4]:
os.environ["GEMINI_API_KEY"] = GEMENI_API_KEY
google_ef = embedding_functions.GoogleGeminiEmbeddingFunction(
    model_name="gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT",
    dimension=768,
)

In [5]:
# Init Chroma Vector Database
chroma_client = chromadb.PersistentClient(path="articles")
collecation_name = "articles"
collection = chroma_client.get_or_create_collection(name=collecation_name, embedding_function=google_ef)
print(collection)

Collection(name=articles)


In [11]:
# First Prompt to Llam LLM
client_groq = Groq(api_key=GROQ_API_KEY)
completion = client_groq.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
      {
        "role": "user",
        "content": "Hello ?"
      }
    ],
    temperature=1,
    max_completion_tokens=1024,
    top_p=1,
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

Hello. How can I assist you today?

In [7]:
# Function to load documents from a directory
def load_documents_from_directory(directory_path):
    documents = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            with open(os.path.join(directory_path, filename), "r", encoding="utf-8") as file:
                documents.append({"id": filename, "text": file.read()})
    return documents

In [8]:
# Function to split text into chunks
def split_text(text, chunk_size=1000, chunk_overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

In [9]:
# Load documents from the directory
directory_path = "./news_articles/"
documents = load_documents_from_directory(directory_path=directory_path)
print(f"Number Of Documents: {len(documents)}")

Number Of Documents: 21


In [29]:
# Split documents into chunks
chunked_documents = []
for doc in documents:
    chunks = split_text(text=doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({"id": f"{doc["id"]}_chunk{i+1}", "text": chunk})
print(chunked_documents)

[{'id': '05-06-amazon-launches-free-channels-check-marks-come-to-gmail-and-openai-raises-more-moolah.txt_chunk1', 'text': 'It’s that time of week again, folks — Week in Review (WiR) time. For those new to the scene, WiR is TechCrunch’s regular newsletter that recaps the biggest tech stories over the past few days. There’s no better digest for the person on the go, we’d argue — but of course, we’re a little biased.\n\nBefore we get into the meat of the thing, a quick reminder that TC City Spotlight: Atlanta is fast approaching. On June 7, TechCrunch is headed to Atlanta, where we’ll host a pitch competition, a talk on the economics of equality, a panel discussion on investing in the Atlanta ecosystem and more.\n\nElsewhere, there’s a TechCrunch Live event with Persona and Index Ventures on May 10, which will touch on how Persona keeps pace with new threats and how Index made a prescient move to spot and back Persona early on. And we have Disrupt in San Francisco from September 19–21 — o

In [41]:
# Function to Generate embeddings for the document chunks - Doc2Vect
def get_gemeni_embedding(text):
    response = gemeni_client.models.embed_content(
        model="gemini-embedding-2",
        contents=text,
        config=types.EmbedContentConfig(
            output_dimensionality=164
        )
)
    embedding = response.embeddings[0].values
    return embedding

In [ ]:
# Get Embedding Vectors
chunked_documents = chunked_documents[0:3]
for doc_chunk in chunked_documents:
    doc_chunk["embedding"] = get_gemeni_embedding(doc_chunk["text"])

In [43]:
# Save Embedding Into Vector Database
for doc_chunk in chunked_documents:
    collection.upsert(ids=[doc_chunk["id"]], documents=[doc_chunk["text"]], embeddings=[doc_chunk["embedding"]])
